In [10]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
CLC Caption Generator (grayscale + Level-3 direct classes, center-weighted)

Computes dominant land-cover composition (original CLC Level-3 codes)
from grayscale CLC_reference tiles, applies center-weighted priority
(human-made → other center → remaining), and produces lowercase,
comma-separated captions (no 'and', no internal commas).

Outputs:
  captions_<size>_level3.parquet
  captions_<size>_level3.csv
"""

import os, re
import numpy as np
import pandas as pd
import rasterio
from scipy.stats import entropy
from IPython.display import clear_output

# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
PATCH_DIR = "/home/ubuntu/SENSERO/GeoTiff/Patch_64"
LEGEND_PATH = "clc_legend.csv"
OUT_PARQUET = os.path.join(PATCH_DIR, "captions_64_level3.parquet")
OUT_CSV     = os.path.join(PATCH_DIR, "captions_64_level3.csv")

# ------------------------------------------------------------------
# LEGEND + CLEANING
# ------------------------------------------------------------------
legend = pd.read_csv(LEGEND_PATH)

def clean_label(text):
    """Remove internal commas and extra spaces, lowercase."""
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text.replace(",", "")).strip().lower()

clc2label3 = {
    int(r["CLC_CODE"]): clean_label(r["LABEL3"])
    for _, r in legend.iterrows()
}

def clc_label_short(code):
    """Return Level-3 label, cleaned and lowercase."""
    try:
        lbl = clc2label3.get(int(code))
        if lbl:
            return lbl
    except Exception:
        pass
    return f"class {code}"

# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------
artificial_classes = {111,112,121,122,123,124,131,132,133}

def extract_base_filename(path):
    """Remove extension and _CLC suffix."""
    name = os.path.splitext(os.path.basename(path))[0]
    name = re.sub(r"_CLC.*$", "", name)
    return name

# ------------------------------------------------------------------
# CENTER-WEIGHTED ANALYSIS
# ------------------------------------------------------------------
def reorder_center_weighted(dominant, center_hist, min_center_pct=1.0):
    """Reorder: human-made in center → other center → rest."""
    center_art = [c for c,p in center_hist.items()
                  if c in artificial_classes and p >= min_center_pct]
    center_non = [c for c,p in center_hist.items()
                  if c not in artificial_classes and p >= min_center_pct]
    center_art.sort(key=lambda c: -center_hist[c])
    center_non.sort(key=lambda c: -center_hist[c])
    center_present = set(center_art + center_non)
    tail = [c for c,_,_ in dominant if c not in center_present]
    c2 = {c:(c,n,p) for c,n,p in dominant}
    ordered = center_art + center_non + tail
    return [c2[c] for c in ordered if c in c2]

def analyze_clc_tif_level3(path, center_frac=0.2):
    """Compute class presence (Level-3, center-weighted)."""
    with rasterio.open(path) as src:
        clc = src.read(1).astype(np.int32)

    h, w = clc.shape
    mask = clc > 0
    if not np.any(mask):
        return "unknown land cover", []

    flat = clc[mask].flatten()
    u, cnts = np.unique(flat, return_counts=True)
    total = cnts.sum()
    dominant = [(int(u[i]), cnts[i], 100 * cnts[i] / total)
                for i in np.argsort(-cnts) if cnts[i] > 0]
    dominant = [t for t in dominant if t[2] >= 2.0]
    if not dominant:
        return "unknown land cover", []

    # --- center square ---
    cside = int(center_frac * min(h, w))
    y0, y1 = h // 2 - cside // 2, h // 2 + cside // 2
    x0, x1 = w // 2 - cside // 2, w // 2 + cside // 2
    mask_center = np.zeros((h, w), bool)
    mask_center[y0:y1, x0:x1] = True

    center_hist = {}
    for code in [d[0] for d in dominant]:
        pct = 100 * np.sum((clc == code) & mask_center) / mask_center.sum()
        if pct > 0:
            center_hist[code] = pct

    dominant = reorder_center_weighted(dominant, center_hist)
    probs = np.array([n for _, n, _ in dominant])
    probs = probs / probs.sum() if probs.sum() > 0 else probs
    ent = entropy(probs, base=2) if probs.sum() > 0 else 0.0

    lines = [f"- {clc_label_short(c)}: {p:.1f}%" for c, _, p in dominant]
    summary = ("Main land cover classes in this patch:\n"
               + "\n".join(lines)
               + f"\n\nEntropy: {ent:.3f}")
    return summary, [int(i) for i in u]

def summary_to_caption(summary):
    blk = re.search(r"Main land cover classes.*?:\n(.*?)\n\n", summary, re.S)
    if not blk:
        return "unknown land cover"
    names = [ln.strip().split(":")[0][2:] for ln in blk.group(1).split("\n")]
    names = [re.sub(",", "", n).strip().lower() for n in names if n.strip()]
    names = list(dict.fromkeys(names))
    return ", ".join(names) if names else "unknown land cover"

# ------------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------------
def main():
    if os.path.exists(OUT_PARQUET):
        df = pd.read_parquet(OUT_PARQUET)
        df = df.drop_duplicates(subset=["BaseFilename"])
        done = set(df["BaseFilename"])
    else:
        df = pd.DataFrame(columns=["BaseFolder","BaseFilename","CLC_codes","Caption_3"])
        done = set()

    clc_root = os.path.join(PATCH_DIR, "CLC_reference")

    for base in sorted(os.listdir(clc_root)):
        base_dir = os.path.join(clc_root, base)
        if not os.path.isdir(base_dir):
            continue

        for f in sorted(os.listdir(base_dir)):
            if not f.lower().endswith(".tif"):
                continue
            clc_path = os.path.join(base_dir, f)
            base_filename = extract_base_filename(clc_path)
            if base_filename in done:
                continue

            try:
                summary, codes = analyze_clc_tif_level3(clc_path)
                caption = summary_to_caption(summary)
            except Exception as e:
                print(f"[WARN] {f}: {e}")
                caption, codes = "unknown land cover", []

            row = {
                "BaseFolder": base,
                "BaseFilename": base_filename,
                "CLC_codes": ", ".join(map(str, codes)),
                "Caption_3": caption
            }

            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
            df = df.drop_duplicates(subset=["BaseFilename"])
            df.to_parquet(OUT_PARQUET, index=False)
            clear_output(wait=False)
            print(f"[INFO] {len(df)} patches → {caption}")

    #df.to_csv(OUT_CSV, index=False)
    #print(f"[DONE] {len(df)} Level-3 captions saved\n  CSV: {OUT_CSV}\n  Parquet: {OUT_PARQUET}")

if __name__ == "__main__":
    main()


[INFO] 10000 patches → pastures, non-irrigated arable land
